In [ ]:
# import subprocess
# import os

# # Define the base directory where you want to save the files
# base_dir = "sam_dictionaries"

# # Ensure the base directory exists
# os.makedirs(base_dir, exist_ok=True)

# # List of URLs to download
# directories = [f"https://baulab.us/u/smarks/autoencoders/pythia-70m-deduped/attn_out_layer{layer}/10_32768/ae.pt" for layer in range(6)]

# for url in directories:
#     # Extract the layer number from the URL for directory naming
#     layer = url.split("attn_out_layer")[1].split("/")[0]
#     # Define the local directory structure
#     local_dir = os.path.join(base_dir, f"attn_out_layer{layer}", "10_32768")
#     # Ensure the local directory structure exists
#     os.makedirs(local_dir, exist_ok=True)
#     # Run wget command, specifying the current directory to save the file
#     subprocess.run(["wget", "-P", local_dir, "--no-parent", url], check=True)


In [ ]:
import torch
from nnsight import LanguageModel
from sam_dictionary import AutoEncoder

ae_list = []

for layer in range(6):
    path = f"./sam_dictionaries/attn_out_layer{layer}/10_32768/ae.pt"

    activation_dim = 512 # dimension of the NN's activations to be autoencoded
    dictionary_size = 64 * activation_dim # number of features in the dictionary
    ae = AutoEncoder(activation_dim, dictionary_size)
    ae.load_state_dict(torch.load(path))
    ae.to("cuda:0")

    ae_list.append(ae)


print("Loaded dictionaries")
        
pythia = LanguageModel("EleutherAI/pythia-70m", device_map="cuda:0", dispatch=True)

print("Loaded GPT")

kwargs = {
    "load_in_4bit": True,
    "device_map": "cuda:0",
    "dispatch": True
}

mixtral = LanguageModel("mistralai/Mixtral-8x7B-Instruct-v0.1", **kwargs)

print("Loaded Mixtral in 4bit")

In [ ]:
with pythia.trace("trace"):
    activation = pythia.gpt_neox.layers[0].attention.output[0].save()

    reconstructed = ae_list[0].encoder(activation)

    reconstructed.save()